In [1]:
"""
--- IMPORTS ---
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

from typing import Optional, overload
from itertools import product
from datetime import datetime
import os

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Imports loaded.")

Using device: cuda
Imports loaded.


In [2]:
"""
--- CONSTANTS ---
"""

GAMMA = 1.4
X_START, X_THROAT, X_LENGTH = 0.0, 0.5, 1.0
X_SHOCK_REF, x_shock = 0.75, 0.5

CLOSE_BD = 1e-6

SHOCK_DET_POINTS = int(1e3)
SHOCK_DET_TRAIN_EPOCHS = int(0.5e4)
SHOCK_DET_LR = 1e-3
SHOCK_DET_PDE_WEIGHT, SHOCK_DET_BD_WEIGHT = 1.0, 1.0
SHOCK_DET_GRAD_CLIP = 5.0
SHOCK_DET_PRINT_EVERY = 250

SHOCK_FORMATION_TOL = 1e-1

DUAL_NET_LR = 1e-4
DUAL_NET_POINTS = int(0.5e3)
DUAL_NET_EPOCHS = int(1e4)
DUAL_NET_DROP_LR = int(0.25 * DUAL_NET_EPOCHS)
DUAL_NET_PDE_WEIGHT = 1.0
DUAL_NET_BD_WEIGHT = 1.0
DUAL_NET_INTERFACE_WEIGHT = 1.0
DUAL_NET_GRAD_CLIP = 5.0
DUAL_NET_PRINT_EVERY = 500

RAR_TRIGGER_EVERY = int(0.5 * DUAL_NET_DROP_LR)
RAR_CANDIDATE_COUNT = DUAL_NET_POINTS
RAR_CANDIDATE_FILTER = 0.5

SHOCK_DET_MODEL_NAME = "shock_det_model"
DUAL_NET_PRE_MODEL_NAME = "dual_net_pre_model"
DUAL_NET_POST_MODEL_NAME = "dual_net_post_model"
INTERFACE_MODEL_NAME = "interface_model"


@overload
def get_area(x: float) -> float: ...
@overload
def get_area(x: torch.Tensor) -> torch.Tensor: ...
def get_area(x: float | torch.Tensor) -> float | torch.Tensor:
    return 1.0 + 2.2 * (x - X_THROAT) ** 2.0


@overload
def get_d_area(x: float) -> float: ...
@overload
def get_d_area(x: torch.Tensor) -> torch.Tensor: ...
def get_d_area(x: float | torch.Tensor) -> float | torch.Tensor:
    return 4.4 * (x - X_THROAT)


print("Constants defined.")

Constants defined.


In [3]:
class ReferenceCSV:
    DEFAULT_FILENAME = "/kaggle/input/datasets/prantikdasiitmds/area-mach-number-ref-csv/area-mach-number-reference.csv"

    def __init__(self, filename: Optional[str] = None):
        filename = filename or self.DEFAULT_FILENAME
        self.reference = pd.read_csv(filename)
        self.subsonic = self.reference[self.reference["M"] < 1.0].sort_values("A")
        self.supersonic = self.reference[self.reference["M"] >= 1.0].sort_values("A")

    def get_values_from_csv(self):
        points = torch.linspace(0, 1, 1000, device=device)[:, None].requires_grad_(True)
        M_ref_sup, M_ref_sub, p_ref_sup, p_ref_sub, t_ref_sup, t_ref_sub = map(
            lambda x: self.get_reference(points, x[0], x[1]),
            product(["M", "p", "T"], ["supersonic", "subsonic"]),
        )

        return points, M_ref_sup, M_ref_sub, p_ref_sup, p_ref_sub, t_ref_sup, t_ref_sub

    def get_reference(self, x_t: torch.Tensor, to: str, branch_name: str):
        area_ratio = get_area(x_t).detach().cpu().numpy()  # type: ignore
        branch = self.supersonic if branch_name == "supersonic" else self.subsonic
        return scipy.interpolate.interp1d(
            branch["A"],
            self.get_branch_data(to, branch),
            bounds_error=False,
            fill_value=(branch[to].iloc[0], branch[to].iloc[-1]),  # type: ignore
        )(area_ratio).squeeze()

    @staticmethod
    def get_branch_data(to: str, branch: pd.DataFrame):
        return branch[to] if to not in ["p", "rho", "T"] else 1 / branch[to]


print("ReferenceCSV data-class defined.")


def _bisection_for_mach_no(a: float, supersonic: bool):
    def func(M: float) -> float:
        t = 1.0 + (GAMMA - 1.0) / 2.0 * M**2.0
        expo = (GAMMA + 1.0) / (2.0 * (GAMMA - 1.0))
        pred = (1.0 / M) * (2.0 / (GAMMA + 1.0) * t) ** expo
        return pred - a

    low, high = (1.0 + CLOSE_BD, 10.0) if supersonic else (CLOSE_BD, 1.0 - CLOSE_BD)

    for _ in range(300):
        mid = (low + high) / 2
        if func(mid) * func(low) < 0:
            high = mid
        else:
            low = mid

    return (low + high) / 2


def get_back_pressure_from_shock(x_shock: float) -> float:
    A_shock = get_area(x_shock)
    M_pre_shock = _bisection_for_mach_no(A_shock, supersonic=True)

    A_exit = get_area(X_LENGTH)
    M_exit = _bisection_for_mach_no(A_exit, supersonic=False)

    p0e_by_p0shock = (
        (((GAMMA + 1.0) * M_pre_shock**2.0) / (2.0 + (GAMMA - 1.0) * M_pre_shock**2.0))
        ** (GAMMA / (GAMMA - 1.0))
    ) * (
        ((GAMMA + 1.0) / (2.0 * GAMMA * M_pre_shock**2.0 - GAMMA + 1.0))
        ** (1.0 / (GAMMA - 1.0))
    )

    return p0e_by_p0shock * (1.0 + (GAMMA - 1.0) / 2.0 * M_exit**2.0) ** (
        -GAMMA / (GAMMA - 1.0)
    )


def get_velocity_at_inlet():
    A_inlet = get_area(X_START)
    M_inlet = _bisection_for_mach_no(A_inlet, supersonic=False)
    T_inlet = 1.0 / (1 + (GAMMA - 1) / 2 * M_inlet**2)
    return M_inlet * np.sqrt(GAMMA * T_inlet)


print("Back pressure function, inlet velocity function defined.")
P_EXIT = get_back_pressure_from_shock(X_SHOCK_REF)
U_INLET = get_velocity_at_inlet()

ReferenceCSV data-class defined.
Back pressure function, inlet velocity function defined.


In [4]:
class DenseSubNet(nn.Module):
    def __init__(
        self, in_dim: int, out_dim: int, hid_dim: int = 128, num_hidden_layers: int = 3
    ):
        super().__init__()
        layers = (
            [nn.Linear(in_dim, hid_dim), nn.Tanh()]
            + [
                layer
                for _ in range(num_hidden_layers)
                for layer in self.make_layer(hid_dim, hid_dim)
            ]
            + [nn.Linear(hid_dim, out_dim)]
        )

        self.net = nn.Sequential(*layers)
        self._init_weights()

    @staticmethod
    def make_layer(in_dim: int, out_dim: int) -> list[nn.Module]:
        return [nn.Linear(in_dim, out_dim), nn.Tanh()]

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        self.net[-1].bias.data[0] = 1.0  # type: ignore
        self.net[-1].bias.data[1] = U_INLET  # type: ignore
        self.net[-1].bias.data[2] = 1.0  # type: ignore

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)  # (x) -> (rho, u, P, T)
        rho = nn.functional.softplus(out[:, 0:1])  # strictly positive
        u = out[:, 1:2]
        p = nn.functional.softplus(out[:, 2:3])  # strictly positive
        return torch.cat([rho, u, p], dim=1)


print("Dense sub-network class defined.")

Dense sub-network class defined.


In [5]:
class Loss:
    @staticmethod
    def get_pde_residuals_nosq(
        model: nn.Module, x: torch.Tensor, causal_weights: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        x = x.requires_grad_(True)
        outputs = model(x)

        rho, u, P = outputs[:, 0], outputs[:, 1], outputs[:, 2]
        A, dA_dx = get_area(x), get_d_area(x)
        H = (1 / 2) * u**2 + GAMMA / (GAMMA - 1.0) * P / rho

        # fmt: off
        mass = Loss.d_(rho * u * A, x)
        momentum = Loss.d_(A * (rho * u**2 + P), x) - P * dA_dx
        energy = Loss.d_(rho * u * H * A, x)
        # fmt: on

        common = torch.cat([mass, momentum, energy], dim=1)
        return common * causal_weights if causal_weights is not None else common

    @staticmethod
    def get_bd_residuals(model: nn.Module) -> torch.Tensor:
        outputs = model(torch.tensor([[X_START]], device=device))
        rho_inlet, u_inlet, P_inlet = outputs[0, 0], outputs[0, 1], outputs[0, 2]
        H = (1 / 2) * u_inlet**2 + GAMMA / (GAMMA - 1.0) * P_inlet / rho_inlet

        isentropic_inlet = torch.square(P_inlet / rho_inlet**GAMMA - 1.0)
        enthalpy_inlet = torch.square(H - (GAMMA / (GAMMA - 1.0)))
        velocity_inlet = torch.square(torch.relu(-1.0 * u_inlet))

        a_inlet = torch.sqrt(GAMMA * P_inlet / rho_inlet)
        supersonic_inlet = torch.square(torch.relu(u_inlet - a_inlet))

        outputs = model(torch.tensor([[X_THROAT]], device=device))
        rho_throat, u_throat, P_throat = outputs[0, 0], outputs[0, 1], outputs[0, 2]

        velocity_throat = torch.square(u_throat**2.0 - GAMMA * P_throat / rho_throat)

        outputs = model(torch.tensor([[X_START + X_LENGTH]], device=device))
        P_exit = outputs[0, 2]

        pressure_exit = torch.square(P_exit - P_EXIT)

        return (
            isentropic_inlet
            + enthalpy_inlet
            + velocity_inlet * 5.0
            + supersonic_inlet
            + velocity_throat * 10.0
            + pressure_exit
        )

    @staticmethod
    def get_left_bd_residuals(model: nn.Module) -> torch.Tensor:
        outputs = model(torch.tensor([[X_START]], device=device))
        rho_inlet, u_inlet, P_inlet = outputs[0, 0], outputs[0, 1], outputs[0, 2]
        H = (1 / 2) * u_inlet**2 + GAMMA / (GAMMA - 1.0) * P_inlet / rho_inlet

        isentropic_inlet = torch.square(P_inlet / rho_inlet**GAMMA - 1.0)
        enthalpy_inlet = torch.square(H - (GAMMA / (GAMMA - 1.0)))
        velocity_inlet = torch.square(torch.relu(-1.0 * u_inlet))

        outputs = model(torch.tensor([[X_THROAT]], device=device))
        rho_throat, u_throat, P_throat = outputs[0, 0], outputs[0, 1], outputs[0, 2]

        velocity_throat = torch.square(u_throat**2.0 - GAMMA * P_throat / rho_throat)

        return isentropic_inlet + enthalpy_inlet + velocity_inlet + velocity_throat
    
    @staticmethod
    def get_right_bd_residuals(model: nn.Module) -> torch.Tensor:
        outputs = model(torch.tensor([[X_START + X_LENGTH]], device=device))
        P_exit = outputs[0, 2]

        pressure_exit = torch.square(P_exit - P_EXIT)

        return pressure_exit

    @staticmethod
    def d_(f: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        return torch.autograd.grad(f.sum(), x, create_graph=True)[0]

    @staticmethod
    def _causal_weights(x: torch.Tensor, epsilon: float = 1.0) -> torch.Tensor:
        return torch.exp(-epsilon * x[:, 0]).unsqueeze(1).detach()

print("Loss class defined.")

Loss class defined.


In [6]:
class ShockParam(nn.Module):
    def __init__(self, x: float):
        super().__init__()
        self.x_shock = nn.Parameter(torch.tensor([x], device=device))

    def forward(self, net_L: DenseSubNet, net_R: DenseSubNet) -> torch.Tensor:
        return get_interface_residuals(self, net_L, net_R)

def get_interface_residuals(
    interface: ShockParam, leftnet: DenseSubNet, rightnet: DenseSubNet
) -> torch.Tensor:
    x_interface = interface.x_shock.view(1, 1).requires_grad_(True)
    outputs_left, outputs_right = leftnet(x_interface), rightnet(x_interface)
    A_interface = get_area(x_interface)
    rho_left, u_left, P_left = (
        outputs_left[0, 0],
        outputs_left[0, 1],
        outputs_left[0, 2],
    )
    rho_right, u_right, P_right = (
        outputs_right[0, 0],
        outputs_right[0, 1],
        outputs_right[0, 2],
    )

    mass_left = rho_left * u_left * A_interface
    mass_right = rho_right * u_right * A_interface
    momentum_left = rho_left * u_left**2.0 * A_interface + P_left * A_interface
    momentum_right = rho_right * u_right**2.0 * A_interface + P_right * A_interface
    H_left = (1 / 2) * u_left**2.0 + GAMMA / (GAMMA - 1.0) * P_left / rho_left
    H_right = (1 / 2) * u_right**2.0 + GAMMA / (GAMMA - 1.0) * P_right / rho_right
    energy_left = rho_left * u_left * H_left * A_interface
    energy_right = rho_right * u_right * H_right * A_interface

    return (
        torch.square(mass_left - mass_right)
        + torch.square(momentum_left - momentum_right)
        + torch.square(energy_left - energy_right)
    )

print("Shock Parameter Class defined.")

Shock Parameter Class defined.


In [7]:
class ModelHandler:
    SAVE_DIR_NAME = "saved_models"

    @staticmethod
    def get_save_path(name: str) -> str:
        parent_dir = os.path.dirname(os.path.abspath("__file__"))
        save_dir = os.path.join(parent_dir, ModelHandler.SAVE_DIR_NAME)
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        model_name = f"{name}.{timestamp}.pt"
        return os.path.join(save_dir, model_name)

    @staticmethod
    def save(model: nn.Module, save_path: str) -> None:
        torch.save(model.state_dict(), save_path)
        print(f"Model saved to {save_path}")

    @staticmethod
    def load(model: nn.Module, load_path: str) -> None:
        if not os.path.exists(load_path):
            raise FileNotFoundError(f"No model found at {load_path}")

        model.load_state_dict(torch.load(load_path, map_location=device))
        print(f"Model loaded from {load_path}")

    @staticmethod
    def update_best_model(
        current_model: nn.Module,
        current_loss: float,
        best_snapshot: dict,
        best_loss: float,
    ) -> tuple[dict, float]:
        if current_loss < best_loss:
            return current_model.state_dict(), current_loss
        return best_snapshot, best_loss

    @staticmethod
    def mk_points(start: float, end: float, num_points: int) -> torch.Tensor:
        points = torch.linspace(start, end, num_points, device=device)
        shuffle_index = torch.randperm(points.shape[0])
        return points[shuffle_index][:, None].requires_grad_(True)

    @staticmethod
    def residual_based_adaptive_refinement(
        old_points: torch.Tensor, new_points: torch.Tensor, model: DenseSubNet
    ) -> torch.Tensor:
        residuals = Loss.get_pde_residuals_nosq(model, new_points)
        residuals = residuals.abs().mean(dim=1).detach()
    
        with torch.no_grad():
            _, indices = torch.topk(
                residuals, int(RAR_CANDIDATE_COUNT * RAR_CANDIDATE_FILTER)
            )
            selected = new_points[indices].detach().requires_grad_(True)
            return torch.cat([old_points.detach(), selected], dim=0).requires_grad_(True)


print("ModelHandler class defined.")

ModelHandler class defined.


In [8]:
"""
--- SHOCK DETECTION ---
"""

def compute_loss_weights(model, points):
    model.train()
    output_layer = model.net[-1]

    # PDE gradient norm
    model.zero_grad()
    cw  = Loss._causal_weights(points)
    res = Loss.get_pde_residuals_nosq(model, points, cw)
    res.square().mean().backward()
    pde_grad_norm = output_layer.weight.grad.norm().item()

    # BC gradient norm
    model.zero_grad()
    bd  = Loss.get_bd_residuals(model)
    bd.mean().backward()
    bc_grad_norm = output_layer.weight.grad.norm().item()

    model.zero_grad()

    # Set weights so PDE and BC contribute equally
    pde_weight = 1.0
    bc_weight  = pde_grad_norm / (bc_grad_norm + 1e-10)

    return pde_weight, bc_weight


def shock_detection_training_loop():
    loss_history, best_state, best_loss = [], {}, float("inf")
    model = DenseSubNet(in_dim=1, out_dim=3).to(device)  # [x -> (rho, u, P)]
    optimizer = torch.optim.Adam(model.parameters())
    points = ModelHandler.mk_points(X_START, X_START + X_LENGTH, SHOCK_DET_POINTS)

    print("Starting shock detection training loop.")
    SHOCK_DET_PDE_WEIGHT, SHOCK_DET_BD_WEIGHT = compute_loss_weights(model, points)
    for epoch in range(SHOCK_DET_TRAIN_EPOCHS):
        model.train()
        optimizer.zero_grad()

        causal_weights = Loss._causal_weights(points)
        pde_residuals = Loss.get_pde_residuals_nosq(model, points, causal_weights)
        bd_residuals = Loss.get_bd_residuals(model)
        loss = (
            SHOCK_DET_PDE_WEIGHT * pde_residuals.square().mean()
            + SHOCK_DET_BD_WEIGHT * bd_residuals.mean()
        )

        loss_history.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=SHOCK_DET_GRAD_CLIP)
        best_state, best_loss = ModelHandler.update_best_model(
            model, loss.item(), best_state, best_loss
        )

        optimizer.step()

        if epoch == 10:
            SHOCK_DET_PDE_WEIGHT, SHOCK_DET_BD_WEIGHT = compute_loss_weights(model, points)

        if epoch % SHOCK_DET_PRINT_EVERY == 0:
            total_loss = f"loss: {loss.item():.6f}"
            pde_loss = f"pde_loss: {pde_residuals.square().mean().item():.6f}"
            bd_loss = f"bd_loss: {bd_residuals.mean().item():.6f}"
            print(f"Epoch {epoch} - {total_loss}, {pde_loss}, {bd_loss}")

    model.load_state_dict(best_state)
    save_path = ModelHandler.get_save_path(SHOCK_DET_MODEL_NAME)
    ModelHandler.save(model, save_path)
    print(f"Shock detection training completed, model saved at {save_path}.")
    return model, loss_history


shock_detection_model, shock_detection_loss_history = shock_detection_training_loop()

Starting shock detection training loop.
Epoch 0 - loss: 2579221.750000, pde_loss: 2089773.750000, bd_loss: 10.113861
Epoch 250 - loss: 68007.843750, pde_loss: 28620.539062, bd_loss: 0.189925
Epoch 500 - loss: 43863.054688, pde_loss: 18979.056641, bd_loss: 0.119990
Epoch 750 - loss: 62491.796875, pde_loss: 35459.906250, bd_loss: 0.130347
Epoch 1000 - loss: 29232.251953, pde_loss: 13745.135742, bd_loss: 0.074679
Epoch 1250 - loss: 29453.132812, pde_loss: 19288.812500, bd_loss: 0.049012
Epoch 1500 - loss: 27754.830078, pde_loss: 15519.710938, bd_loss: 0.058998
Epoch 1750 - loss: 63827.937500, pde_loss: 43367.679688, bd_loss: 0.098659
Epoch 2000 - loss: 35535.277344, pde_loss: 28883.037109, bd_loss: 0.032077
Epoch 2250 - loss: 26810.369141, pde_loss: 17615.267578, bd_loss: 0.044339
Epoch 2500 - loss: 44330.433594, pde_loss: 29514.949219, bd_loss: 0.071440
Epoch 2750 - loss: 19403.167969, pde_loss: 13880.681641, bd_loss: 0.026629
Epoch 3000 - loss: 22500.171875, pde_loss: 19221.417969, bd_l

In [9]:
shock_detection_model.eval()


def predict_shock_formation():
    points = ModelHandler.mk_points(X_START, X_START + X_LENGTH, SHOCK_DET_POINTS)

    # check @ remove
    scalar_bd_loss = Loss.get_bd_residuals(shock_detection_model).item()
    print(f"Boundary loss for shock detection model: {scalar_bd_loss:.6f}")

    u = shock_detection_model(points)[:, 1]
    du_dx_lin = Loss.d_(u, points).squeeze().detach().cpu().numpy()
    x_lin = points.squeeze().detach().cpu().numpy()

    du_dx_lin[x_lin <= X_THROAT] = 0.0
    minima_index = int(np.argmin(du_dx_lin))
    minima_slope = du_dx_lin[minima_index]
    if minima_slope >= 0:
        print("No shock detected, using initial predicted shock location.")
        return x_shock

    change_threshold, change_index = SHOCK_FORMATION_TOL * minima_slope, minima_index
    for i in range(minima_index, 0, -1):
        if du_dx_lin[i] > change_threshold:
            change_index = i
            break

    return np.clip(x_lin[change_index], X_THROAT, X_START + X_LENGTH)


x_shock = predict_shock_formation()
print(f"Predicted shock location: {x_shock:.6f}")

Boundary loss for shock detection model: 0.005757
Predicted shock location: 0.898899


In [ ]:
"""
--- SUB-NETWORK LOOP ---
"""


def dual_network_training_loop():
    pre_shock_model = DenseSubNet(in_dim=1, out_dim=3).to(device)
    post_shock_model = DenseSubNet(in_dim=1, out_dim=3).to(device)
    pre_shock_model.load_state_dict(shock_detection_model.state_dict())
    post_shock_model.load_state_dict(shock_detection_model.state_dict())
    shock_param = ShockParam(x_shock).to(device)

    points = ModelHandler.mk_points(X_START, X_START + X_LENGTH, SHOCK_DET_POINTS)
    DUAL_NET_PDE_WEIGHT, DUAL_NET_BD_WEIGHT = compute_loss_weights(shock_detection_model, points)

    all_params = (
        list(pre_shock_model.parameters())
        + list(post_shock_model.parameters())
        + list(shock_param.parameters())
    )

    loss_history = []
    optimizer = torch.optim.Adam(all_params, lr=DUAL_NET_LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=DUAL_NET_DROP_LR, gamma=0.5)

    pre_points = ModelHandler.mk_points(X_START, x_shock, DUAL_NET_POINTS)
    post_points = ModelHandler.mk_points(x_shock, X_START + X_LENGTH, DUAL_NET_POINTS)
    pre_shock_best_state, pre_shock_best_loss = {}, float("inf")
    post_shock_best_state, post_shock_best_loss = {}, float("inf")
    interface_best_state, interface_best_loss = {}, float("inf")

    for epoch in range(DUAL_NET_EPOCHS):
        pre_shock_model.train()
        post_shock_model.train()
        shock_param.train()
        optimizer.zero_grad()

        causal_weights_pre = Loss._causal_weights(pre_points)
        causal_weights_post = Loss._causal_weights(post_points)

        pde_residuals_pre = Loss.get_pde_residuals_nosq(
            pre_shock_model, pre_points, causal_weights_pre
        )
        pde_residuals_post = Loss.get_pde_residuals_nosq(
            post_shock_model, post_points, causal_weights_post
        )
        pde_loss = (
            pde_residuals_pre.square().mean() + pde_residuals_post.square().mean()
        )

        bd_residuals_pre = Loss.get_left_bd_residuals(pre_shock_model)
        bd_residuals_post = Loss.get_right_bd_residuals(post_shock_model)
        bd_loss = bd_residuals_pre.mean() + bd_residuals_post.mean()

        interface_loss = shock_param(pre_shock_model, post_shock_model).mean()

        loss = (
            DUAL_NET_PDE_WEIGHT * pde_loss
            + DUAL_NET_BD_WEIGHT * bd_loss
            + DUAL_NET_INTERFACE_WEIGHT * interface_loss
        )

        loss_history.append(loss.item())
        loss.backward()
        nn.utils.clip_grad_norm_(all_params, max_norm=DUAL_NET_GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        pre_shock_best_state, pre_shock_best_loss = ModelHandler.update_best_model(
            pre_shock_model, loss.item(), pre_shock_best_state, pre_shock_best_loss
        )
        post_shock_best_state, post_shock_best_loss = ModelHandler.update_best_model(
            post_shock_model, loss.item(), post_shock_best_state, post_shock_best_loss
        )
        interface_best_state, interface_best_loss = ModelHandler.update_best_model(
            shock_param, loss.item(), interface_best_state, interface_best_loss
        )

        if epoch % DUAL_NET_PRINT_EVERY == 0:
            new_points_pre = ModelHandler.mk_points(X_START, x_shock, DUAL_NET_POINTS)
            new_points_post = ModelHandler.mk_points(
                x_shock, X_START + X_LENGTH, DUAL_NET_POINTS
            )
            pre_points = ModelHandler.residual_based_adaptive_refinement(
                pre_points, new_points_pre, pre_shock_model
            )
            post_points = ModelHandler.residual_based_adaptive_refinement(
                post_points, new_points_post, post_shock_model
            )

        if epoch % DUAL_NET_PRINT_EVERY == 0:
            shock_at = f"x_shock: {shock_param.x_shock.item():.6f}"
            total_loss = f"loss: {loss.item():.6f}"
            pde_loss_pre = f"pde_loss_pre: {pde_residuals_pre.square().mean().item():.6f}"
            pde_loss_post = f"pde_loss_post: {pde_residuals_post.mean().item():.6f}"
            interface_loss_t = f"interface_loss: {interface_loss.item():.6f}"
            bd_loss_pre = f"bd_loss_pre: {bd_residuals_pre.mean().item():.6f}"
            bd_loss_post = f"bd_loss_post: {bd_residuals_post.mean().item():.6f}"
            print(
                f"Epoch {epoch} - {shock_at}, {total_loss}, {pde_loss_pre}, {pde_loss_post}, "
                f"{interface_loss_t}, {bd_loss_pre}, {bd_loss_post}"
            )

    pre_shock_model.load_state_dict(pre_shock_best_state)
    post_shock_model.load_state_dict(post_shock_best_state)
    shock_param.load_state_dict(interface_best_state)

    pre_shock_save_path = ModelHandler.get_save_path(DUAL_NET_PRE_MODEL_NAME)
    post_shock_save_path = ModelHandler.get_save_path(DUAL_NET_POST_MODEL_NAME)
    interface_save_path = ModelHandler.get_save_path(INTERFACE_MODEL_NAME)

    ModelHandler.save(pre_shock_model, pre_shock_save_path)
    ModelHandler.save(post_shock_model, post_shock_save_path)
    ModelHandler.save(shock_param, interface_save_path)

    print(
        f"Shock detection training completed, models saved at {pre_shock_save_path}, {post_shock_save_path}, {interface_save_path}."
    )
    return pre_shock_model, post_shock_model, shock_param, loss_history

pre_shock_model, post_shock_model, shock_param, dual_net_loss_history = dual_network_training_loop()

Epoch 0 - x_shock: 0.898899, loss: 68007.148438, pde_loss_pre: 5108.463867, pde_loss_post: -219.255173, interface_loss: 0.000000, bd_loss_pre: 0.001985, bd_loss_post: 0.002224
Epoch 500 - x_shock: 0.856424, loss: 9226.498047, pde_loss_pre: 8720.596680, pde_loss_post: 0.130723, interface_loss: 2.791889, bd_loss_pre: 0.000224, bd_loss_post: 0.000000
Epoch 1000 - x_shock: 0.825936, loss: 13454.103516, pde_loss_pre: 12324.238281, pde_loss_post: 12.780831, interface_loss: 2.298016, bd_loss_pre: 0.000434, bd_loss_post: 0.000000
Epoch 1500 - x_shock: 0.799244, loss: 17414.529297, pde_loss_pre: 15705.798828, pde_loss_post: 8.471107, interface_loss: 2.614990, bd_loss_pre: 0.000745, bd_loss_post: 0.000000
Epoch 2000 - x_shock: 0.768848, loss: 21738.972656, pde_loss_pre: 18034.173828, pde_loss_post: 38.025562, interface_loss: 2.766006, bd_loss_pre: 0.001033, bd_loss_post: 0.000000
Epoch 2500 - x_shock: 0.740585, loss: 23218.283203, pde_loss_pre: 20381.199219, pde_loss_post: -0.211021, interface_l

In [ ]:
post_shock_model.eval()
pre_shock_model.eval()
shock_param.eval()

shock_at = shock_param.x_shock.item()
print(f"Final predicted shock location: {shock_at:.6f}")

x_eval = torch.linspace(X_START, X_START + X_LENGTH, 1000, device=device)[:, None]
post_shock = x_eval[:, 0] > shock_at
pre_shock_points, post_shock_points = x_eval[~post_shock], x_eval[post_shock]
pre_shock_points_np = pre_shock_points.squeeze().cpu().numpy()
post_shock_points_np = post_shock_points.squeeze().cpu().numpy()

with torch.no_grad():
    outputs_pre = pre_shock_model(pre_shock_points)
    outputs_post = post_shock_model(post_shock_points)

    rho_pre, u_pre, P_pre = outputs_pre[:, 0], outputs_pre[:, 1], outputs_pre[:, 2]
    rho_post, u_post, P_post = (
        outputs_post[:, 0],
        outputs_post[:, 1],
        outputs_post[:, 2],
    )

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(dual_net_loss_history, label="Dual Network Training Loss")  # type: ignore
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Dual Network Training Loss History")
plt.legend()
plt.grid()
plt.show()

In [ ]:
x_eval_np = x_eval.squeeze().cpu().numpy()
r = np.sqrt(get_area(x_eval).detach().cpu().numpy() / np.pi)

plt.figure(figsize=(12, 8))
plt.subplot(3, 1, 1)
plt.plot(x_eval_np, r, label="Nozzle Radius")
plt.plot(x_eval_np, -r, label="Nozzle Radius (mirrored)")
plt.title("Nozzle Shape")
plt.xlabel("x")
plt.ylabel("Radius")
plt.legend()

M_pre = u_pre / torch.sqrt(GAMMA * P_pre / rho_pre)
M_post = u_post / torch.sqrt(GAMMA * P_post / rho_post)
M_pre, M_post = M_pre.cpu().numpy(), M_post.cpu().numpy()
P_pre_np, P_post_np = P_pre.cpu().numpy(), P_post.cpu().numpy()

reference = ReferenceCSV()
points, M_ref_sup, M_ref_sub, p_ref_sup, p_ref_sub, t_ref_sup, t_ref_sub = (
    reference.get_values_from_csv()
)
points = points.squeeze().detach().cpu().numpy()

plt.subplot(3, 1, 2)
plt.plot(pre_shock_points_np.squeeze(), M_pre, label="Mach Number", color="blue")
plt.plot(post_shock_points_np.squeeze(), M_post, label="Mach Number", color="blue")
plt.axvline(x=shock_at, color="red", linestyle="--", label="Shock Location")
plt.axvline(
    x=X_SHOCK_REF, color="green", linestyle="--", label="Reference Shock Location"
)
plt.plot(points, M_ref_sup, label="Reference Supersonic", color="orange")
plt.plot(points, M_ref_sub, label="Reference Subsonic", color="orange")
plt.title("Mach Number Distribution")
plt.xlabel("x")
plt.ylabel("Mach Number")
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(pre_shock_points_np.squeeze(), P_pre_np, label="Pressure", color="blue")
plt.plot(post_shock_points_np.squeeze(), P_post_np, label="Pressure", color="blue")
plt.axvline(x=shock_at, color="red", linestyle="--", label="Shock Location")
plt.axvline(
    x=X_SHOCK_REF, color="green", linestyle="--", label="Reference Shock Location"
)
plt.plot(points, p_ref_sup, label="Reference Supersonic", color="orange")
plt.plot(points, p_ref_sub, label="Reference Subsonic", color="orange")
plt.title("Pressure Distribution")
plt.xlabel("x")
plt.ylabel("Pressure")
plt.legend()

plt.tight_layout()
plt.show()